# Random Variables, Independence &Expected Value

## Independence vs Dependence
**Two events A and B are independent if knowing that one occurred gives you zero information about whether the other occurred. Formally, the joint probability equals the product of marginals:

Independent: P(A ∩ B) = P(A) · P(B)
Equivalently: P(A|B) = P(A) — conditioning on B doesn't change the probability of A at all.

Events are dependent when knowledge of one changes the probability of the other. P(A|B) ≠ P(A). The dependency can be positive (knowing B makes A more likely) or negative (knowing B makes A less likely).

Dependent: P(A ∩ B) ≠ P(A) · P(B)**

> Independent: Flipping a coin, then rolling a die. The coin result tells you nothing about what the die will show.
> 
> 
> 
> Dependent: Drawing two cards without replacement. Once you know the first card is an Ace, the probability of the second being an Ace changes (3/51 instead of 4/52).

* The Naive Bayes classifier assumes all features are independent. The Chi-Square test is used in feature selection to check if a feature is independent of the target — if it is, the feature carries no useful information and can be dropped. Correlation matrices also measure linear dependence between numerical features.*

In [1]:
import numpy as np
import pandas as pd
from scipy import stats

# ─── Independence Test ────────────────────────────────────────
# The formal test: P(A ∩ B) == P(A) * P(B) ?

# --- SCENARIO 1: Truly Independent Events ---
# Flip a coin AND roll a die — completely separate
np.random.seed(42)
n = 100_000

coin = np.random.choice(['H', 'T'], n)          # Heads or Tails
die  = np.random.choice(range(1,7), n)            # 1-6

p_heads   = np.mean(coin == 'H')
p_six     = np.mean(die == 6)
p_both    = np.mean((coin == 'H') & (die == 6))
p_product = p_heads * p_six                      # what it SHOULD be if independent

print("═══ SCENARIO 1: Coin + Die (Independent) ═══")
print(f"P(Heads)            = {p_heads:.4f}   (expected 0.5000)")
print(f"P(Six)              = {p_six:.4f}   (expected 0.1667)")
print(f"P(Heads AND Six)    = {p_both:.4f}   (observed)")
print(f"P(Heads) * P(Six)   = {p_product:.4f}   (expected if independent)")
print(f"Independent? {'✓ YES' if abs(p_both - p_product) < 0.002 else '✗ NO'}")
print()

# --- SCENARIO 2: Clearly Dependent Events ---
# Weather: Cloudy and Rainy are DEPENDENT
# If it's cloudy, probability of rain is much higher

days = 100_000
# Base rates
p_cloudy_true = 0.40    # 40% of days are cloudy
p_rain_given_cloudy  = 0.60  # 60% chance rain if cloudy
p_rain_given_clear   = 0.05  # only 5% chance rain if clear

cloudy = np.random.random(days) < p_cloudy_true
rain   = np.where(
    cloudy,
    np.random.random(days) < p_rain_given_cloudy,
    np.random.random(days) < p_rain_given_clear
)

p_c    = np.mean(cloudy)
p_r    = np.mean(rain)
p_both = np.mean(cloudy & rain)
p_prod = p_c * p_r

print("═══ SCENARIO 2: Cloudy & Rain (Dependent) ═══")
print(f"P(Cloudy)              = {p_c:.4f}")
print(f"P(Rain)                = {p_r:.4f}")
print(f"P(Cloudy AND Rain)     = {p_both:.4f}  (observed)")
print(f"P(Cloudy) * P(Rain)    = {p_prod:.4f}  (expected IF independent)")
print(f"Difference             = {abs(p_both-p_prod):.4f}  ← should be ~0 if independent")
print(f"Independent? {'✓ YES' if abs(p_both - p_prod) < 0.002 else '✗ NO — clearly dependent!'}")
print()

# --- P(A|B) check: does knowing B change A? ---
p_rain_given_cloudy_obs  = p_both / p_c
print(f"P(Rain | Cloudy)  = {p_rain_given_cloudy_obs:.4f}  ← very high")
print(f"P(Rain) overall   = {p_r:.4f}  ← much lower")
print(f"→ Knowing it's cloudy CHANGES rain probability: dependent!")
print()

# ─── Chi-Square Test: Statistical independence test ───────────
# Used in data science to check if two categorical variables are independent
print("─── Chi-Square Independence Test ───")

# Create contingency table
ct = pd.crosstab(
    pd.Series(cloudy.astype(int), name='Cloudy'),
    pd.Series(rain.astype(int),   name='Rain')
)

chi2, p_value, dof, expected = stats.chi2_contingency(ct)
print(f"Chi-square statistic : {chi2:,.2f}")
print(f"p-value             : {p_value:.6f}")
print(f"Conclusion: {'DEPENDENT (reject independence)' if p_value < 0.05 else 'INDEPENDENT'}")
print(f"(p < 0.05 means we reject the hypothesis that they are independent)")

═══ SCENARIO 1: Coin + Die (Independent) ═══
P(Heads)            = 0.4994   (expected 0.5000)
P(Six)              = 0.1654   (expected 0.1667)
P(Heads AND Six)    = 0.0825   (observed)
P(Heads) * P(Six)   = 0.0826   (expected if independent)
Independent? ✓ YES

═══ SCENARIO 2: Cloudy & Rain (Dependent) ═══
P(Cloudy)              = 0.4000
P(Rain)                = 0.2694
P(Cloudy AND Rain)     = 0.2394  (observed)
P(Cloudy) * P(Rain)    = 0.1078  (expected IF independent)
Difference             = 0.1316  ← should be ~0 if independent
Independent? ✗ NO — clearly dependent!

P(Rain | Cloudy)  = 0.5985  ← very high
P(Rain) overall   = 0.2694  ← much lower
→ Knowing it's cloudy CHANGES rain probability: dependent!

─── Chi-Square Independence Test ───
Chi-square statistic : 36,682.86
p-value             : 0.000000
Conclusion: DEPENDENT (reject independence)
(p < 0.05 means we reject the hypothesis that they are independent)


# Random Variables

** A random variable is a function that assigns a numerical value to each outcome in a sample space. Instead of dealing with abstract events like "heads" or "rainy day", we convert them to numbers so we can do mathematics on them.

There are two fundamental types**
## Discrete Random Variable
**Countable values
Takes on a finite or countably infinite set of values. You can list them: 0, 1, 2, 3... There are gaps between possible values.

Examples:
• # of heads in 10 flips (0–10)
• # of customers/hour
• # of defects in a batch
• Die roll result (1–6)**

## Continuous Random Variable
**Uncountable values
Takes on any value in an interval — infinitely many possible values. No gaps. You can never list all possibilities.

Examples:
• Height of a person (any real #)
• Time until server responds
• Stock price change
• Temperature readings**

*For discrete variables, you can ask "P(X = 5)" and get a meaningful non-zero answer. For continuous variables, P(X = exactly 5.000...) = 0 because there are infinite possible values. Instead you ask "P(4 < X < 6)".*

# Discrete Variables
# Probability Mass Function (PMF)

**The PMF tells you the exact probability that a discrete random variable X equals a specific value x. It assigns a probability "mass" to each possible outcome.

PMF: P(X = x) = f(x)     where Σ f(x) = 1 for all x
Two rules must always hold: (1) every probability must be ≥ 0, and (2) all probabilities must sum to exactly 1.**

***Scenario: You roll a fair 6-sided die. X = the number shown. What's the PMF? What about the PMF for X = number of heads in 3 coin flips?***

*PMFs are used in Naive Bayes for discrete features (word counts in text), in A/B test analysis (counting successes), and in anomaly detection (flagging events with very low P(X=k) under a Poisson model, like unusually high server error rates).*

In [3]:
from scipy.stats import binom, poisson, geom
import numpy as np

# ─── PMF: Probability Mass Function ──────────────────────────

# EXAMPLE 1: Fair Die (Uniform Discrete PMF)
print("═══ PMF Example 1: Fair Die ═══")
outcomes = [1,2,3,4,5,6]
pmf_die  = {x: 1/6 for x in outcomes}

print(f"{'X':<6} {'P(X=x)':<12} {'Bar'}")
print("-" * 35)
for x, p in pmf_die.items():
    bar = '█' * int(p * 60)
    print(f"{x:<6} {p:<12.4f} {bar}")
print(f"Sum of all probabilities = {sum(pmf_die.values()):.1f} ✓")
print()

# EXAMPLE 2: Binomial PMF — # of Heads in 3 coin flips
# X ~ Binomial(n=3, p=0.5)
# P(X=k) = C(3,k) * 0.5^k * 0.5^(3-k)
print("═══ PMF Example 2: Binomial(n=3, p=0.5) — Heads in 3 flips ═══")
n, p = 3, 0.5

print(f"{'Heads (k)':<12} {'P(X=k)':<14} {'Bar'}")
print("-" * 50)
for k in range(n+1):
    prob = binom.pmf(k, n, p)
    bar  = '█' * int(prob * 80)
    print(f"{k:<12} {prob:<14.4f} {bar}")
print()

# EXAMPLE 3: Poisson PMF — website hits per minute
# λ = average rate (mean = variance for Poisson!)
print("═══ PMF Example 3: Poisson(λ=4) — Website hits/minute ═══")
lam = 4   # average 4 hits per minute

print(f"{'Hits (k)':<12} {'P(X=k)':<14} {'Bar'}")
print("-" * 55)
for k in range(13):
    prob = poisson.pmf(k, lam)
    bar  = '█' * int(prob * 100)
    print(f"{k:<12} {prob:<14.4f} {bar}")
print()

# Key PMF queries
print("─── Answering questions with PMF ───")
print(f"P(exactly 4 hits/min)  = {poisson.pmf(4, 4):.4f}")
print(f"P(exactly 0 hits/min)  = {poisson.pmf(0, 4):.4f}")
print(f"P(more than 6 hits)    = {1 - poisson.cdf(6, 4):.4f}  ← uses CDF")
print(f"P(between 2 and 5)     = {sum(poisson.pmf(k,4) for k in range(2,6)):.4f}")

═══ PMF Example 1: Fair Die ═══
X      P(X=x)       Bar
-----------------------------------
1      0.1667       ██████████
2      0.1667       ██████████
3      0.1667       ██████████
4      0.1667       ██████████
5      0.1667       ██████████
6      0.1667       ██████████
Sum of all probabilities = 1.0 ✓

═══ PMF Example 2: Binomial(n=3, p=0.5) — Heads in 3 flips ═══
Heads (k)    P(X=k)         Bar
--------------------------------------------------
0            0.1250         █████████
1            0.3750         ██████████████████████████████
2            0.3750         ██████████████████████████████
3            0.1250         ██████████

═══ PMF Example 3: Poisson(λ=4) — Website hits/minute ═══
Hits (k)     P(X=k)         Bar
-------------------------------------------------------
0            0.0183         █
1            0.0733         ███████
2            0.1465         ██████████████
3            0.1954         ███████████████████
4            0.1954         ███████████████

# Continuous Variables
# Probability Density Function (PDF)

**For continuous variables, P(X = exactly x) = 0 for any specific value. Instead, the PDF f(x) describes the density of probability around each point. Probability is only meaningful over an interval:

P(a ≤ X ≤ b) = ∫[a to b] f(x) dx
The PDF f(x) itself can be greater than 1 — it's a density, not a probability. What must equal 1 is the total area under the curve:

∫[-∞ to +∞] f(x) dx = 1
Think of it like a "probability per unit length" — the higher the curve at a point, the more probability is packed in that region.**

*Human adult heights follow a Normal distribution with mean μ=170cm and σ=10cm. What fraction of people are between 160cm and 180cm? What about below 150cm?*

In [4]:
from scipy.stats import norm, expon, uniform
from scipy import integrate
import numpy as np

# ─── PDF: Probability Density Function ───────────────────────

# EXAMPLE 1: Normal Distribution
# X ~ Normal(μ=170, σ=10)  — human heights in cm
print("═══ PDF Example 1: Normal Distribution — Heights ═══")
mu, sigma = 170, 10
dist = norm(mu, sigma)

# KEY POINT: f(x) is NOT a probability, it's a density!
x_val = 170
density_at_mean = dist.pdf(x_val)
print(f"f(170) = {density_at_mean:.4f}  ← density, NOT probability (can be > 1)")
print(f"P(X = exactly 170cm) = 0  ← always zero for continuous!")
print()

# Probability requires an INTERVAL (integrating the density)
p_160_to_180 = dist.cdf(180) - dist.cdf(160)
p_below_150  = dist.cdf(150)
p_above_190  = 1 - dist.cdf(190)
p_1_sigma    = dist.cdf(mu+sigma) - dist.cdf(mu-sigma)   # 68-95-99.7 rule
p_2_sigma    = dist.cdf(mu+2*sigma) - dist.cdf(mu-2*sigma)
p_3_sigma    = dist.cdf(mu+3*sigma) - dist.cdf(mu-3*sigma)

print(f"P(160 ≤ X ≤ 180)  = {p_160_to_180:.4f}  ({p_160_to_180*100:.1f}%)")
print(f"P(X < 150cm)       = {p_below_150:.4f}  ({p_below_150*100:.2f}%)")
print(f"P(X > 190cm)       = {p_above_190:.4f}  ({p_above_190*100:.2f}%)")
print(f"P(within 1σ)       = {p_1_sigma:.4f}  ← 68-95-99.7 rule")
print(f"P(within 2σ)       = {p_2_sigma:.4f}")
print(f"P(within 3σ)       = {p_3_sigma:.4f}")
print()

# EXAMPLE 2: Exponential Distribution (time between events)
# X ~ Exponential(rate λ=0.5) — avg wait time = 1/λ = 2 minutes
print("═══ PDF Example 2: Exponential — Wait Time ═══")
rate = 0.5
exp_dist = expon(scale=1/rate)   # scipy uses scale=1/λ

print(f"Average wait time     = {1/rate:.1f} minutes")
print(f"P(wait < 1 min)       = {exp_dist.cdf(1):.4f}")
print(f"P(wait < 2 min)       = {exp_dist.cdf(2):.4f}")
print(f"P(wait > 5 min)       = {1 - exp_dist.cdf(5):.4f}")
print(f"P(1 min < wait < 3)   = {exp_dist.cdf(3) - exp_dist.cdf(1):.4f}")
print()

# EXAMPLE 3: Verify total area = 1
print("─── Verify: Total area under Normal PDF = 1 ───")
total_area, error = integrate.quad(dist.pdf, -np.inf, np.inf)
print(f"∫ f(x) dx from -∞ to +∞ = {total_area:.10f} ← exactly 1 ✓")

# ASCII visualization of Normal PDF
print()
print("─── Normal PDF Shape (ASCII) ───")
x_range = np.linspace(140, 200, 61)
for x in x_range[::3]:
    density = dist.pdf(x)
    bar = '▓' * int(density * 300)
    print(f"  {x:5.0f}cm | {bar}")

═══ PDF Example 1: Normal Distribution — Heights ═══
f(170) = 0.0399  ← density, NOT probability (can be > 1)
P(X = exactly 170cm) = 0  ← always zero for continuous!

P(160 ≤ X ≤ 180)  = 0.6827  (68.3%)
P(X < 150cm)       = 0.0228  (2.28%)
P(X > 190cm)       = 0.0228  (2.28%)
P(within 1σ)       = 0.6827  ← 68-95-99.7 rule
P(within 2σ)       = 0.9545
P(within 3σ)       = 0.9973

═══ PDF Example 2: Exponential — Wait Time ═══
Average wait time     = 2.0 minutes
P(wait < 1 min)       = 0.3935
P(wait < 2 min)       = 0.6321
P(wait > 5 min)       = 0.0821
P(1 min < wait < 3)   = 0.3834

─── Verify: Total area under Normal PDF = 1 ───
∫ f(x) dx from -∞ to +∞ = 0.0000000000 ← exactly 1 ✓

─── Normal PDF Shape (ASCII) ───
    140cm | 
    143cm | 
    146cm | 
    149cm | ▓
    152cm | ▓▓
    155cm | ▓▓▓
    158cm | ▓▓▓▓▓
    161cm | ▓▓▓▓▓▓▓
    164cm | ▓▓▓▓▓▓▓▓▓
    167cm | ▓▓▓▓▓▓▓▓▓▓▓
    170cm | ▓▓▓▓▓▓▓▓▓▓▓
    173cm | ▓▓▓▓▓▓▓▓▓▓▓
    176cm | ▓▓▓▓▓▓▓▓▓
    179cm | ▓▓▓▓▓▓▓
    182cm | ▓▓▓▓▓



# Both Types
# Cumulative Distribution Function (CDF)

**The CDF F(x) answers: "What is the probability that X is less than or equal to x?" It accumulates probability as x increases — hence "cumulative". It works for both discrete AND continuous random variables.

CDF: F(x) = P(X ≤ x)
For discrete: F(x) = Σ P(X = k) for all k ≤ x    (sum up the PMF)

For continuous: F(x) = ∫[-∞ to x] f(t) dt    (integrate the PDF)

Key CDF properties: F(-∞) = 0, F(+∞) = 1, and F is always non-decreasing.**

*Why CDF is so powerful: Any probability question can be answered with just the CDF.
P(a ≤ X ≤ b) = F(b) − F(a)    P(X > a) = 1 − F(a)*

*Critical insight: The inverse CDF (called ppf in scipy) is used to build confidence intervals, find critical values for hypothesis tests, and compute percentile-based statistics. When a data scientist says "the 95th percentile of response times", they're computing dist.ppf(0.95).*

In [5]:
from scipy.stats import binom, norm
import numpy as np

# ─── CDF ──────────────────────────────────────────────────────

# PART 1: Discrete CDF — Binomial(n=10, p=0.3)
# Scenario: 10 product recommendations, 30% click rate
# X = number of clicks
print("═══ Discrete CDF: Binomial(n=10, p=0.3) — Clicks ═══")
n, p = 10, 0.3

print(f"{'k':<5} {'P(X=k) PMF':<16} {'F(k)=P(X≤k) CDF':<20} {'CDF bar'}")
print("-" * 70)
for k in range(n+1):
    pmf_val = binom.pmf(k, n, p)
    cdf_val = binom.cdf(k, n, p)
    bar = '░' * int(cdf_val * 30)
    print(f"{k:<5} {pmf_val:<16.4f} {cdf_val:<20.4f} {bar}")
print()

# CDF answers all probability questions in one shot
print("─── Answering questions with Discrete CDF ───")
print(f"P(X ≤ 3 clicks)    = F(3)         = {binom.cdf(3, n, p):.4f}")
print(f"P(X > 5 clicks)    = 1 - F(5)     = {1-binom.cdf(5, n, p):.4f}")
print(f"P(2 ≤ X ≤ 6)       = F(6) - F(1)  = {binom.cdf(6,n,p)-binom.cdf(1,n,p):.4f}")
print(f"P(X = 4 exactly)   = F(4) - F(3)  = {binom.cdf(4,n,p)-binom.cdf(3,n,p):.4f}")
print()

# PART 2: Continuous CDF — Normal
print("═══ Continuous CDF: Normal(μ=0, σ=1) — Standard Normal ═══")
std_norm = norm(0, 1)

print(f"{'x':<8} {'F(x) = P(Z ≤ x)':<22} {'CDF bar'}")
print("-" * 60)
for x in [-3, -2, -1, -0.5, 0, 0.5, 1, 2, 3]:
    cdf_val = std_norm.cdf(x)
    bar = '▓' * int(cdf_val * 30)
    print(f"{x:<8} {cdf_val:<22.4f} {bar}")
print()

# Inverse CDF (Quantile Function / Percent Point Function)
# Given probability, find the x value → essential for confidence intervals!
print("─── Inverse CDF (Quantile Function) ───")
print("'What value has X% of the distribution below it?'")
print()
for q in [0.05, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]:
    z = std_norm.ppf(q)
    print(f"  {q*100:.0f}th percentile (z) = {z:+.4f}")
print()
print("─── This is how confidence intervals are built! ───")
z_95 = std_norm.ppf(0.975)   # two-tailed 95% CI
print(f"95% CI uses z = ±{z_95:.4f}  (the famous 1.96)")

═══ Discrete CDF: Binomial(n=10, p=0.3) — Clicks ═══
k     P(X=k) PMF       F(k)=P(X≤k) CDF      CDF bar
----------------------------------------------------------------------
0     0.0282           0.0282               
1     0.1211           0.1493               ░░░░
2     0.2335           0.3828               ░░░░░░░░░░░
3     0.2668           0.6496               ░░░░░░░░░░░░░░░░░░░
4     0.2001           0.8497               ░░░░░░░░░░░░░░░░░░░░░░░░░
5     0.1029           0.9527               ░░░░░░░░░░░░░░░░░░░░░░░░░░░░
6     0.0368           0.9894               ░░░░░░░░░░░░░░░░░░░░░░░░░░░░░
7     0.0090           0.9984               ░░░░░░░░░░░░░░░░░░░░░░░░░░░░░
8     0.0014           0.9999               ░░░░░░░░░░░░░░░░░░░░░░░░░░░░░
9     0.0001           1.0000               ░░░░░░░░░░░░░░░░░░░░░░░░░░░░░
10    0.0000           1.0000               ░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░

─── Answering questions with Discrete CDF ───
P(X ≤ 3 clicks)    = F(3)         = 0.6496
P(X > 


# Summary Statistics
# Expected Value E(X)

**The expected value E(X) — also called the mean or μ — is the long-run average of a random variable over many repetitions. It's the "center of gravity" of the probability distribution.

Discrete: E(X) = Σ x · P(X=x)
Continuous: E(X) = ∫ x · f(x) dx
It does NOT need to be a value X can actually take. The expected value of a die roll is 3.5 — and you can never actually roll a 3.5.**

In [6]:
from scipy.stats import binom, norm, poisson
import numpy as np

# ─── Expected Value ───────────────────────────────────────────

# EXAMPLE 1: Manual calculation — Weighted Die
# A loaded die: 6 comes up 50% of the time, others equal
print("═══ E(X) Example 1: Weighted Die ═══")
outcomes = np.array([1, 2, 3, 4, 5, 6])
# 5 faces share remaining 50%, each gets 10%
probs    = np.array([0.10, 0.10, 0.10, 0.10, 0.10, 0.50])

print(f"Sum of probabilities = {probs.sum():.2f} ✓")

# E(X) = Σ x * P(X=x)
ev_loaded = np.sum(outcomes * probs)
ev_fair   = np.sum(outcomes * (1/6))

print(f"\nFair die:   E(X) = {ev_fair:.4f}  (1*1/6 + 2*1/6 + ... + 6*1/6)")
print(f"Loaded die: E(X) = {ev_loaded:.4f}  (biased toward 6)")
print()

# EXAMPLE 2: Expected value of common distributions
print("═══ E(X) of Common Distributions ═══")
print(f"{'Distribution':<30} {'Formula':<20} {'E(X)'}")
print("-" * 65)
print(f"{'Uniform(1,6) — fair die':<30} {'(a+b)/2':<20} {(1+6)/2:.4f}")
print(f"{'Binomial(n=10, p=0.3)':<30} {'n*p':<20} {10*0.3:.4f}")
print(f"{'Poisson(λ=4)':<30} {'λ':<20} {4:.4f}")
print(f"{'Normal(μ=170, σ=10)':<30} {'μ':<20} {170:.4f}")
print(f"{'Exponential(λ=0.5)':<30} {'1/λ':<20} {1/0.5:.4f}")
print()

# EXAMPLE 3: Real-world E(X) — Gambling game
print("═══ E(X) Real World: Gambling Expected Profit ═══")
print("You pay $5 to play. Roll a die:")
print("  Roll 6    → win $20 (profit = +$15)")
print("  Roll 4/5  → win $5  (profit = $0, break even)")
print("  Roll 1/2/3 → lose $5 (profit = -$5)")
print()

payouts = np.array([-5, -5, -5, 0, 0, 15])   # profit per outcome
probs_fair = np.array([1/6] * 6)

ev_game = np.sum(payouts * probs_fair)
print(f"E(profit) = Σ outcome * P(outcome)")
for payout, prob in zip(payouts, probs_fair):
    print(f"  ${payout:+d} × {prob:.4f} = ${payout*prob:+.4f}")
print(f"  {'─'*28}")
print(f"  E(profit) = ${ev_game:+.4f} per game")
print(f"  Over 1000 games: ~${ev_game*1000:+.0f} expected total")
print()

# Verify with simulation
np.random.seed(0)
rolls = np.random.randint(1, 7, 100_000)
profits = np.where(rolls==6, 15, np.where(rolls>=4, 0, -5))
print(f"Simulation (100k games): avg profit = ${profits.mean():+.4f}  (expected: ${ev_game:.4f})")

═══ E(X) Example 1: Weighted Die ═══
Sum of probabilities = 1.00 ✓

Fair die:   E(X) = 3.5000  (1*1/6 + 2*1/6 + ... + 6*1/6)
Loaded die: E(X) = 4.5000  (biased toward 6)

═══ E(X) of Common Distributions ═══
Distribution                   Formula              E(X)
-----------------------------------------------------------------
Uniform(1,6) — fair die        (a+b)/2              3.5000
Binomial(n=10, p=0.3)          n*p                  3.0000
Poisson(λ=4)                   λ                    4.0000
Normal(μ=170, σ=10)            μ                    170.0000
Exponential(λ=0.5)             1/λ                  2.0000

═══ E(X) Real World: Gambling Expected Profit ═══
You pay $5 to play. Roll a die:
  Roll 6    → win $20 (profit = +$15)
  Roll 4/5  → win $5  (profit = $0, break even)
  Roll 1/2/3 → lose $5 (profit = -$5)

E(profit) = Σ outcome * P(outcome)
  $-5 × 0.1667 = $-0.8333
  $-5 × 0.1667 = $-0.8333
  $-5 × 0.1667 = $-0.8333
  $+0 × 0.1667 = $+0.0000
  $+0 × 0.1667 = $+0.0000


# Spread
# Variance Var(X) & Std Dev σ

**While E(X) tells you the center, Var(X) tells you the spread — how far values typically deviate from the mean. It's the average squared distance from the mean:

Var(X) = E[(X − μ)²] = E[X²] − (E[X])²
Standard deviation σ = √Var(X) brings it back to the original units. Variance uses squared units (cm², $²), which is often less intuitive. The standard deviation is usually what you report.

σ = √Var(X)     (same units as X)
Key computational shortcut: Var(X) = E[X²] − (E[X])² — the mean of the squares minus the square of the mean.**

In [7]:
import numpy as np
from scipy.stats import binom, norm, poisson

# ─── Variance & Standard Deviation ───────────────────────────

# EXAMPLE 1: Manual variance calculation
print("═══ Var(X) Example 1: Fair Die ═══")
x    = np.array([1,2,3,4,5,6])
probs = np.array([1/6]*6)

mu      = np.sum(x * probs)                     # E(X) = 3.5
var_def = np.sum((x - mu)**2 * probs)           # E[(X-μ)²] — definition
var_alt = np.sum(x**2 * probs) - mu**2          # E[X²] - (E[X])² — shortcut
std_dev = np.sqrt(var_def)

print(f"E(X)                   = {mu:.4f}")
print(f"Var(X) via definition  = {var_def:.4f}")
print(f"Var(X) via shortcut    = {var_alt:.4f}  ← same result ✓")
print(f"Std Dev σ              = {std_dev:.4f}")
print(f"Interpretation: rolls are typically ±{std_dev:.2f} from the mean of {mu}")
print()

# EXAMPLE 2: Why variance matters — two investments with same E(X)
print("═══ Var(X) Example 2: Same Mean, Different Risk ═══")
print("Two investments with SAME expected return but DIFFERENT variance:")
print()

np.random.seed(42)
n = 10_000

# Conservative: returns cluster tightly around mean
conservative = np.random.normal(loc=5, scale=2, size=n)   # μ=5%, σ=2%

# Aggressive: same mean, much higher spread
aggressive    = np.random.normal(loc=5, scale=15, size=n) # μ=5%, σ=15%

for name, data in [('Conservative', conservative), ('Aggressive', aggressive)]:
    print(f"  {name}:")
    print(f"    Mean (E(X))     = {data.mean():+.2f}%")
    print(f"    Variance        = {data.var():.2f}")
    print(f"    Std Dev (σ)     = {data.std():.2f}%")
    print(f"    P(lose money)   = {(data < 0).mean():.2%}")
    print(f"    P(gain > 20%)   = {(data > 20).mean():.2%}")
    print()

# EXAMPLE 3: Variance of common distributions
print("═══ Variance of Common Distributions ═══")
print(f"{'Distribution':<30} {'E(X)':<12} {'Var(X)':<15} {'σ (Std Dev)'}")
print("-" * 72)
dists = [
    ("Binomial(n=10, p=0.3)", 10*0.3,  10*0.3*0.7),  # Var = n*p*(1-p)
    ("Poisson(λ=4)",           4,      4),           # Var = λ (same as mean!)
    ("Normal(μ=170, σ=10)",   170,    100),         # Var = σ²
    ("Exponential(λ=0.5)",    2,      4),           # Var = 1/λ²
    ("Uniform(1,6)",           3.5,   (6-1)**2/12),  # Var = (b-a)²/12
]
for name, mean, var in dists:
    print(f"{name:<30} {mean:<12.4f} {var:<15.4f} {var**0.5:.4f}")

═══ Var(X) Example 1: Fair Die ═══
E(X)                   = 3.5000
Var(X) via definition  = 2.9167
Var(X) via shortcut    = 2.9167  ← same result ✓
Std Dev σ              = 1.7078
Interpretation: rolls are typically ±1.71 from the mean of 3.5

═══ Var(X) Example 2: Same Mean, Different Risk ═══
Two investments with SAME expected return but DIFFERENT variance:

  Conservative:
    Mean (E(X))     = +5.00%
    Variance        = 4.03
    Std Dev (σ)     = 2.01%
    P(lose money)   = 0.66%
    P(gain > 20%)   = 0.00%

  Aggressive:
    Mean (E(X))     = +5.20%
    Variance        = 225.43
    Std Dev (σ)     = 15.01%
    P(lose money)   = 36.46%
    P(gain > 20%)   = 16.14%

═══ Variance of Common Distributions ═══
Distribution                   E(X)         Var(X)          σ (Std Dev)
------------------------------------------------------------------------
Binomial(n=10, p=0.3)          3.0000       2.1000          1.4491
Poisson(λ=4)                   4.0000       4.0000          2.0000



# Transformations
# Linear Transformation Properties
**
When you apply a linear transformation Y = aX + b to a random variable, the mean and variance change in predictable, elegant ways. These rules are the foundation of standardization (z-scores), unit conversion, and combining random variables.

E(aX + b) = a·E(X) + b     (mean scales and shifts)
Var(aX + b) = a²·Var(X)     (variance scales by a², NOT affected by b)
σ(aX + b) = |a|·σ(X)       (std dev scales by |a|)
Notice that adding a constant b shifts the mean but does NOT change variance — shifting all values by the same amount doesn't change how spread out they are.**

*Z-score standardization is a linear transformation that converts any normal to Standard Normal(0,1). This is why we can use a single z-table for all normal distributions. The rule Var(X+Y) = Var(X)+Var(Y) (for independent X,Y) is the foundation of the Central Limit Theorem — the most important theorem in statistics.*

In [8]:
import numpy as np
from scipy.stats import norm

# ─── Linear Transformations ───────────────────────────────────

# EXAMPLE 1: Temperature conversion — Fahrenheit to Celsius
# F = (9/5) * C + 32  →  C = (F - 32) * (5/9)
print("═══ Linear Transformation Example 1: Temperature ═══")
print("X = temp in °C ~ Normal(μ=20, σ=5)")
print("Y = temp in °F = (9/5)*X + 32")
print()

mu_c, sigma_c = 20, 5
a, b = 9/5, 32

# Theoretical transformation
mu_f    = a * mu_c + b                 # E(aX+b) = a*E(X) + b
var_f   = a**2 * sigma_c**2            # Var(aX+b) = a²*Var(X)
sigma_f = np.sqrt(var_f)              # σ(aX+b) = |a|*σ(X)

print(f"Celsius:    E(X) = {mu_c}°C,  σ = {sigma_c}°C,  Var = {sigma_c**2}")
print(f"Fahrenheit: E(Y) = {mu_f}°F,  σ = {sigma_f}°F,  Var = {var_f}")
print()

# Verify with simulation
np.random.seed(0)
celsius = np.random.normal(mu_c, sigma_c, 100_000)
fahrenheit = (9/5) * celsius + 32
print(f"Simulated Fahrenheit mean : {fahrenheit.mean():.4f}  (theory: {mu_f})")
print(f"Simulated Fahrenheit std  : {fahrenheit.std():.4f}   (theory: {sigma_f})")
print()

# EXAMPLE 2: Standardization (Z-score) — the most important transformation!
# Z = (X - μ) / σ  →  a = 1/σ, b = -μ/σ
print("═══ Linear Transformation Example 2: Z-Score Standardization ═══")
print("Z = (X - μ) / σ  →  transforms ANY normal to Standard Normal(0,1)")
print()

# Heights: X ~ Normal(170, 10)
mu_h, sigma_h = 170, 10
heights = np.random.normal(mu_h, sigma_h, 100_000)

# Standardize
z_scores = (heights - mu_h) / sigma_h     # Z = (X - μ) / σ

print(f"Original heights:  mean={heights.mean():.2f}, std={heights.std():.2f}")
print(f"Z-scores:          mean={z_scores.mean():.4f}, std={z_scores.std():.4f}")
print(f"→ Z ~ Normal(0, 1) always after standardization ✓")
print()

# Theoretical proof using transformation rules:
print("─── Theoretical proof: E(Z) and Var(Z) ───")
print(f"Z = (1/σ)*X + (-μ/σ)  →  a=1/σ={1/sigma_h:.3f}, b=-μ/σ={-mu_h/sigma_h:.3f}")
print(f"E(Z) = a*E(X) + b = {1/sigma_h:.3f}*{mu_h} + ({-mu_h/sigma_h:.3f}) = {1/sigma_h*mu_h - mu_h/sigma_h:.4f}")
print(f"Var(Z) = a²*Var(X)  = {(1/sigma_h)**2:.4f} * {sigma_h**2} = {(1/sigma_h)**2 * sigma_h**2:.4f}")
print()

# EXAMPLE 3: Combining independent RVs (sum and difference)
print("═══ Combining Independent Random Variables ═══")
print("If X and Y are INDEPENDENT:")
print("  E(X + Y) = E(X) + E(Y)        (always true)")
print("  Var(X+Y) = Var(X) + Var(Y)    (only if independent!)")
print("  Var(X-Y) = Var(X) + Var(Y)    (variances always ADD)")
print()

# Commute time example
# Train: X ~ Normal(30, 5²), Bus: Y ~ Normal(20, 8²)
mu_x, var_x = 30, 25  # train: avg 30min, σ=5
mu_y, var_y = 20, 64  # bus: avg 20min, σ=8

# Total commute Z = X + Y
mu_z  = mu_x + mu_y
var_z = var_x + var_y    # INDEPENDENT so variances add
std_z = np.sqrt(var_z)

print(f"Train X: E(X)={mu_x}min, σ={var_x**0.5:.1f}min")
print(f"Bus   Y: E(Y)={mu_y}min, σ={var_y**0.5:.1f}min")
print()
print(f"Total Z = X + Y:")
print(f"  E(Z) = E(X)+E(Y) = {mu_x}+{mu_y} = {mu_z}min")
print(f"  Var(Z) = Var(X)+Var(Y) = {var_x}+{var_y} = {var_z}")
print(f"  σ(Z)  = √{var_z} = {std_z:.4f}min  ← NOT σ_x + σ_y! ({var_x**0.5+var_y**0.5:.1f})")
print()
print("⚠ KEY: Standard deviations do NOT add. Variances add (for independent vars).")

# Verify with simulation
np.random.seed(42)
train_times = np.random.normal(mu_x, var_x**0.5, 100_000)
bus_times   = np.random.normal(mu_y, var_y**0.5, 100_000)
total_times = train_times + bus_times

print()
print(f"Simulated total: mean={total_times.mean():.2f}min, σ={total_times.std():.4f}min")
print(f"Theoretical:     mean={mu_z}min,    σ={std_z:.4f}min ✓")

═══ Linear Transformation Example 1: Temperature ═══
X = temp in °C ~ Normal(μ=20, σ=5)
Y = temp in °F = (9/5)*X + 32

Celsius:    E(X) = 20°C,  σ = 5°C,  Var = 25
Fahrenheit: E(Y) = 68.0°F,  σ = 9.0°F,  Var = 81.0

Simulated Fahrenheit mean : 68.0142  (theory: 68.0)
Simulated Fahrenheit std  : 8.9761   (theory: 9.0)

═══ Linear Transformation Example 2: Z-Score Standardization ═══
Z = (X - μ) / σ  →  transforms ANY normal to Standard Normal(0,1)

Original heights:  mean=170.05, std=9.99
Z-scores:          mean=0.0051, std=0.9987
→ Z ~ Normal(0, 1) always after standardization ✓

─── Theoretical proof: E(Z) and Var(Z) ───
Z = (1/σ)*X + (-μ/σ)  →  a=1/σ=0.100, b=-μ/σ=-17.000
E(Z) = a*E(X) + b = 0.100*170 + (-17.000) = 0.0000
Var(Z) = a²*Var(X)  = 0.0100 * 100 = 1.0000

═══ Combining Independent Random Variables ═══
If X and Y are INDEPENDENT:
  E(X + Y) = E(X) + E(Y)        (always true)
  Var(X+Y) = Var(X) + Var(Y)    (only if independent!)
  Var(X-Y) = Var(X) + Var(Y)    (variances al

# Master Summary Cheatsheet

## Concept	        Key Formula	        Type	        DS Application
Independence	P(A∩B) = P(A)·P(B)	Both	        Naive Bayes, feature selection, Chi-square test
Dependence	    P(A∩B) ≠ P(A)·P(B)	Both	        Correlated features, sequential data, without-replacement sampling
PMF	            P(X=x) = f(x), Σf=1	Discrete	    Word counts, click counts, defect counts, Binomial/Poisson models
PDF	            P(a≤X≤b) = ∫f(x)dx	Continuous	    Height/weight, prices, time, Normal/Exponential distributions
CDF	            F(x) = P(X≤x)	    Both	        Percentiles, confidence intervals, hypothesis test p-values
E(X)	        Σ x·P(x) or ∫x·f(x)dx	Both	    Expected return, average loss, long-run averages in RL
Var(X)	        E[X²] − (E[X])²	    Both	        Risk, feature scaling, model uncertainty, bias-variance tradeoff
E(aX+b)	        a·E(X) + b	        Transform	    Unit conversion, scaling rewards, shifting features
Var(aX+b)	    a²·Var(X)	        Transform	    Z-scores, standardization, why adding constant doesn't change spread
Var(X+Y)	    Var(X)+Var(Y) if indep.	Combined	Central Limit Theorem, error propagation, portfolio variance


*These three topics form an unbreakable chain. Independence determines whether you can multiply probabilities (and whether Var(X+Y) = Var(X)+Var(Y)). Random variables with their PMF/PDF/CDF give you the full probability model. E(X) and Var(X) summarize that model into two interpretable numbers — the center and the spread. Together, they are the language in which virtually every machine learning algorithm is written.*